# Scratch Notebook

This notebook is for ad-hoc prototyping and quick checks while developing the optimization workflow.

**Notes**
- Keep experiments short and self-contained.
- Move stable code paths into `src/` modules once validated.


In [ ]:
import random

import matplotlib.pyplot as plt
import networkx as nx
import pyomo.environ as pyo

In [ ]:
# Generate a random digraph
network = nx.gn_graph(40)

# Plot the graph in a small window
plt.figure(figsize=(2, 2))

# Draw the graph without labels and small nodes
nx.draw(network, with_labels=False, node_size=10, node_color="blue", alpha=0.5)
plt.show()

# Print the successors and predecessors of a random node
node = random.choice(list(network.nodes))
print(f"Node {node} has successors: {list(network.successors(node))}")
print(f"Node {node} has predecessors: {list(network.predecessors(node))}")

In [ ]:
# Generate non-negative random times for the nodes in the network
scenario = {node: random.randint(0, 20) for node in network.nodes}
# scenario

In [ ]:
# Extracting network information
no_of_nodes = network.number_of_nodes()
nodes = pyo.Set(initialize=list(network.nodes))

# Setting up the optimization problem
model = pyo.ConcreteModel()

# Activities start-time variables (integer)
model.s = pyo.Var(nodes, domain=pyo.NonNegativeIntegers)

In [ ]:
# Network flow model constraints


def network_constraints_rule(
    model: pyo.ConcreteModel,
    node: int,
    successor: int,
) -> object:
    """Enforce precedence for one arc in the project network."""
    return model.s[successor] >= model.s[node] + scenario[node]


model.network_constraints = pyo.Constraint(
    [
        (node, successor)
        for node in network.nodes
        for successor in network.successors(node)
    ],
    rule=network_constraints_rule,
)


@model.Constraint(model.s, network.successors(model.s))
def flow_constraints(
    model: pyo.ConcreteModel,
    node: int,
    successor: int,
) -> object:
    """Alternative callback form used for quick constraint experiments."""
    return model.s[successor] >= model.s[node] + scenario[node]